# Day 6 Demo 0 - Bedrock from first principles: one call -> the agent loop

The simplest on-ramp, **before** Demo 1. It builds the **agent loop** and its common **variations**, one idea per cell. Each section has:
- a short **pre-flight** (what to set up / check, with the commands), and
- a **works / breaks / why** note grounded in how Bedrock actually evolved - the gotchas you only learn by hitting them.

Ladder here: one call -> the old `invoke_model` (and why `converse` replaced it) -> multi-turn -> the minimal tool loop -> loop variations (guard, streaming, stop reasons, retries). Demo 1 then builds workflows and multi-agent on top.

> `OFFLINE_DEMO = True` runs against a faithful local stub (no AWS) so you can dry-run the shapes. For a real proof-of-concept keep it **False** (the default).

## Set up once, in this order

1. **Enable model access** - Bedrock console -> **Model catalog** -> enable **Anthropic Claude Sonnet 4.5** in **us-west-2**. (The standalone "Model access" page has been folded into the Model catalog.)
2. **Credentials** - `aws configure`, or environment variables, or an attached role.
   ```
   aws sts get-caller-identity
   ```
3. **Inference profiles are not optional for new models.** Newer Claude models are served through a cross-Region **inference profile**; the bare model id is rejected for on-demand use (`ValidationException`). That is why `MODEL` below starts with `us.`.
   ```
   aws bedrock list-inference-profiles --region us-west-2 --query "inferenceProfileSummaries[?contains(inferenceProfileId, 'sonnet-4-5')].inferenceProfileId" --output text
   ```

In [ ]:
# ---- configuration: EDIT THESE ----
OFFLINE_DEMO = False   # False = real Bedrock (default). True = offline stub for dry-runs only.
REGION = "us-west-2"
MODEL  = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"   # <-- a us. inference profile you have ENABLED

import json

In [2]:
# ---- client + a faithful offline stub (shapes match the real bedrock-runtime API) ----
class _Body:
    def __init__(self, payload): self._b = json.dumps(payload).encode()
    def read(self): return self._b

class _Stub:
    """Offline stand-in. Not a real model - it only reproduces the response SHAPES."""
    def converse(self, modelId=None, messages=None, toolConfig=None, system=None, inferenceConfig=None, **kw):
        if toolConfig:
            done = sum(1 for m in messages for c in m.get("content", [])
                       if isinstance(c, dict) and "toolResult" in c)
            names = [t["toolSpec"]["name"] for t in toolConfig["tools"]]
            plan = [(n, {"pnr": "JX48Q2"}, "tu_%d" % (k + 1)) for k, n in enumerate(names)]
            if done < len(plan):
                n, i, tid = plan[done]
                return {"output": {"message": {"role": "assistant",
                            "content": [{"toolUse": {"toolUseId": tid, "name": n, "input": i}}]}},
                        "stopReason": "tool_use", "usage": {"inputTokens": 100, "outputTokens": 20}}
        last = messages[-1]["content"][-1].get("text", "") if messages else ""
        return {"output": {"message": {"role": "assistant",
                    "content": [{"text": "[offline stub] " + last[:60]}]}},
                "stopReason": "end_turn", "usage": {"inputTokens": 100, "outputTokens": 20}}

    def converse_stream(self, modelId=None, messages=None, system=None, inferenceConfig=None, **kw):
        def gen():
            yield {"messageStart": {"role": "assistant"}}
            for chunk in ["An inference profile ", "routes one model call ", "across Regions for capacity."]:
                yield {"contentBlockDelta": {"delta": {"text": chunk}, "contentBlockIndex": 0}}
            yield {"contentBlockStop": {"contentBlockIndex": 0}}
            yield {"messageStop": {"stopReason": "end_turn"}}
            yield {"metadata": {"usage": {"inputTokens": 12, "outputTokens": 9, "totalTokens": 21},
                                "metrics": {"latencyMs": 100}}}
        return {"stream": gen()}

    def invoke_model(self, modelId=None, body=None, **kw):
        return {"body": _Body({"content": [{"type": "text", "text": "[offline stub] reply via invoke_model"}],
                               "stop_reason": "end_turn", "usage": {"input_tokens": 12, "output_tokens": 9}})}

_client = None
def get_client():
    global _client
    if OFFLINE_DEMO:
        return _Stub()
    if _client is None:
        import boto3
        _client = boto3.client("bedrock-runtime", region_name=REGION)
    return _client

print("mode:", "OFFLINE stub" if OFFLINE_DEMO else ("REAL Bedrock - " + REGION))

mode: OFFLINE stub


## Check first: can you actually reach a model?

A 30-second pre-flight saves an hour of confusion. It checks identity, then makes one tiny call and translates the three failures everyone hits on day one.

In [3]:
def preflight():
    if OFFLINE_DEMO:
        print("OFFLINE: skipping live checks. Set OFFLINE_DEMO=False to run the real pre-flight.")
        return
    import boto3
    from botocore.exceptions import ClientError
    try:
        ident = boto3.client("sts", region_name=REGION).get_caller_identity()
        print("identity OK:", ident["Arn"])
    except Exception as e:
        print("CREDENTIALS problem ->", e); return
    try:
        r = get_client().converse(modelId=MODEL,
                                  messages=[{"role": "user", "content": [{"text": "ping"}]}],
                                  inferenceConfig={"maxTokens": 5})
        print("model OK ->", r["output"]["message"]["content"][0]["text"][:40])
    except ClientError as e:
        code = e.response["Error"]["Code"]
        why = {"ValidationException": "model id likely needs a us. inference profile, or the model is not enabled",
               "AccessDeniedException": "enable the model in Model catalog, or fix the IAM policy",
               "ThrottlingException": "your account quota may be 0 (new accounts) - request an increase"}
        print("MODEL problem ->", code, "->", why.get(code, "read the message"))

preflight()

OFFLINE: skipping live checks. Set OFFLINE_DEMO=False to run the real pre-flight.


**The three day-one failures, and why**
- `AccessDeniedException` - the model is not enabled for your account/Region. Fix it in **Model catalog**.
- `ValidationException` - usually a **bare model id** for a model that is on-demand-only via an **inference profile**. Use the `us.` id.
- `ThrottlingException` - a new account often has an **applied quota of 0** for a model. Request an increase; do not just retry forever.

*Evolution note:* the old per-model "Model access" request page is gone; access now lives in the **Model catalog**.

## 1 - One call (the Converse API)

The smallest useful thing: one request, one reply. No memory, no tools, no loop.

In [ ]:
def ask(prompt, system=None, temperature=0.2, max_tokens=512):
    kw = {"modelId": MODEL, "messages": [{"role": "user", "content": [{"text": prompt}]}],
          "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature}}
    if system: kw["system"] = [{"text": system}]
    r = get_client().converse(**kw)
    return r["output"]["message"]["content"][0]["text"]

print(ask("In one sentence, what is an inference profile in Amazon Bedrock?"))

**Works / breaks / why**
- One uniform call shape for every model - that is the whole point of **Converse** (launched **May 30, 2024**). Before it, you wrote a different JSON body per model family.
- Breaks if `MODEL` is a bare id for a profile-only model -> `ValidationException`. The `us.` prefix is the fix.
- `temperature=0` for anything you want to be repeatable; higher only for creative text.

## 2 - The old way: `invoke_model` (and why `converse` replaced it)

`invoke_model` still exists and is model-specific: you send the model's own JSON. For Claude that means `anthropic_version` + `messages`. Useful history, and still needed for the occasional model-only parameter.

In [ ]:
def ask_invoke_model(prompt, max_tokens=256):
    body = {
        "anthropic_version": "bedrock-2023-05-31",     # Claude-on-Bedrock contract (confirm current)
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }
    resp = get_client().invoke_model(modelId=MODEL, body=json.dumps(body))
    data = json.loads(resp["body"].read())             # body is a stream; .read() then json.loads
    return data["content"][0]["text"]

print(ask_invoke_model("Say hello in one short sentence."))

**The evolution (why this matters)**
- **2023:** Bedrock shipped with only `InvokeModel`. Every family had its own body - Claude originally took a single `prompt` string framed with `\n\nHuman:` / `\n\nAssistant:` (text-completions), later the **Messages** format (`anthropic_version`, `messages`, `max_tokens`); Titan took `inputText`. You learned each one.
- **May 30, 2024:** **Converse** unified the request/response across models and added **tool use**, **system prompts**, **streaming**, and **Guardrails** in one place.
- **Today** there are three API families: **Invoke** (`InvokeModel`, `InvokeModelWithResponseStream`, `AsyncInvoke`), **Converse** (`Converse`, `ConverseStream`), and an **OpenAI-compatible** family (`ChatCompletions`, `Responses`).
- *Rule of thumb:* default to **Converse**. Reach for `invoke_model` only for a model-specific parameter not yet surfaced in Converse - and even then, `additionalModelRequestFields` often covers it.

## 3 - Multi-turn (you manage the `messages` list)

The model has **no memory** between calls. A conversation is just the growing `messages` list you send each time.

In [ ]:
def chat_turn(messages, user_text, system=None):
    messages.append({"role": "user", "content": [{"text": user_text}]})
    kw = {"modelId": MODEL, "messages": messages, "inferenceConfig": {"maxTokens": 256, "temperature": 0.2}}
    if system: kw["system"] = [{"text": system}]
    resp = get_client().converse(**kw)
    messages.append(resp["output"]["message"])          # keep the assistant turn for the next round
    return resp["output"]["message"]["content"][0]["text"]

history = []
print(chat_turn(history, "My PNR is JX48Q2."))
print(chat_turn(history, "What was my PNR again?"))      # only works because we resent the history
print("turns stored:", len(history))

**Works / breaks / why**
- Bedrock is **stateless** - if you do not resend the history, the model "forgets". The memory is your list, not the service.
- The history grows every turn, so **input tokens (and cost) grow** with the conversation. Long chats need trimming or summarisation.
- Breaks if you forget to append the **assistant** turn before the next user turn - the thread loses its own last answer.
- *Evolution:* the **Messages** format (roles + content blocks) replaced the original single-string `Human:/Assistant:` prompt.

## 4 - The agent loop (one tool)

This is the core idea of the whole course. The model can now **ask for a tool**: you run it, hand back the result, and repeat until it stops. One tool keeps it simple.

In [ ]:
def lookup_booking(pnr):
    return {"pnr": pnr, "status": "CANCELLED", "flight": "AI-302", "date": "2026-06-12"}
TOOLS = {"lookup_booking": lookup_booking}

TOOL_SPECS = [{"toolSpec": {"name": "lookup_booking", "description": "Look up a booking by its PNR.",
    "inputSchema": {"json": {"type": "object",
        "properties": {"pnr": {"type": "string", "description": "6-char PNR"}}, "required": ["pnr"]}}}}]
SYSTEM = "You are TravelMind. Use the tool to look up a booking before answering. Never invent a PNR."

def agent_loop(user_text, max_steps=5):
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    resp = get_client().converse(modelId=MODEL, system=[{"text": SYSTEM}], messages=messages,
                                 toolConfig={"tools": TOOL_SPECS},
                                 inferenceConfig={"maxTokens": 512, "temperature": 0.2})
    steps = 0
    while resp["stopReason"] == "tool_use" and steps < max_steps:
        steps += 1
        tu = resp["output"]["message"]["content"][-1]["toolUse"]
        messages.append(resp["output"]["message"])                       # 1) assistant tool_use turn FIRST
        out = TOOLS[tu["name"]](**tu["input"])                           # 2) run the tool
        print("  step", steps, "->", tu["name"], tu["input"], "->", out)
        messages.append({"role": "user", "content": [{"toolResult": {    # 3) result, matching id
            "toolUseId": tu["toolUseId"], "content": [{"json": out}]}}]})
        resp = get_client().converse(modelId=MODEL, system=[{"text": SYSTEM}], messages=messages,
                                     toolConfig={"tools": TOOL_SPECS},
                                     inferenceConfig={"maxTokens": 512, "temperature": 0.2})
    
    return resp["output"]["message"]["content"][0]["text"]

print(agent_loop("What is the status of PNR JX48Q2?"))

  step 1 -> lookup_booking {'pnr': 'JX48Q2'} -> {'pnr': 'JX48Q2', 'status': 'CANCELLED', 'flight': 'AI-302', 'date': '2026-06-12'}
[offline stub] 


**The gotchas, and why**
- **Append the assistant `tool_use` message BEFORE the `toolResult`.** This is the single most common error: a `toolResult` with no preceding `toolUse` is rejected. Order is user -> assistant(tool_use) -> user(toolResult).
- **Echo the exact `toolUseId`.** A mismatch is rejected.
- **The tool runs in *your* code** - Bedrock returns a request, it does not call your function. With Converse/InvokeModel this is always client-side tool calling.
- *Evolution:* what everyone called "function calling" in 2023-24 is standardised on Bedrock as **tool use** via `toolConfig`.

## 5 - Variations of the loop

The loop above is the skeleton. Four things turn it from a demo into something you can ship.

### 5a - The guard (the one line that saves you)

A model can keep asking for tools. Without a cap, a bad prompt or a flaky tool becomes an **infinite loop** - which, on a hosted runtime, becomes a **timeout** and a bill.

In [ ]:
def agent_loop_guarded(user_text, max_steps=3):
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    resp = get_client().converse(modelId=MODEL, system=[{"text": SYSTEM}], messages=messages,
                                 toolConfig={"tools": TOOL_SPECS}, inferenceConfig={"maxTokens": 512})
    steps = 0
    while resp["stopReason"] == "tool_use":
        if steps >= max_steps:                                   # <-- the guard
            return "Stopped: hit the step limit (" + str(max_steps) + "). Investigate the prompt or tools."
        steps += 1
        tu = resp["output"]["message"]["content"][-1]["toolUse"]
        messages.append(resp["output"]["message"])
        out = TOOLS[tu["name"]](**tu["input"])
        messages.append({"role": "user", "content": [{"toolResult": {
            "toolUseId": tu["toolUseId"], "content": [{"json": out}]}}]})
        resp = get_client().converse(modelId=MODEL, system=[{"text": SYSTEM}], messages=messages,
                                     toolConfig={"tools": TOOL_SPECS}, inferenceConfig={"maxTokens": 512})
    return resp["output"]["message"]["content"][0]["text"]

print(agent_loop_guarded("Status of PNR JX48Q2?"))

**Why** - the guard is non-negotiable in production. The classic failure: an agent that keeps calling a tool that never satisfies it, until the request times out. Cap the steps; also watch `max_tokens` (next-but-one) so a truncated turn does not masquerade as progress.

### 5b - Streaming (`converse_stream`)

For anything long, stream the tokens as they arrive. Better UX, and it avoids client/idle timeouts on long generations (which matters once the agent is hosted).

In [ ]:
def stream_answer(prompt):
    resp = get_client().converse_stream(modelId=MODEL,
                                        messages=[{"role": "user", "content": [{"text": prompt}]}],
                                        inferenceConfig={"maxTokens": 256})
    stop, usage = None, None
    for event in resp["stream"]:
        if "contentBlockDelta" in event:
            print(event["contentBlockDelta"]["delta"].get("text", ""), end="")
        elif "messageStop" in event:
            stop = event["messageStop"]["stopReason"]
        elif "metadata" in event:
            usage = event["metadata"].get("usage")
    print("\n[stopReason:", stop, "| usage:", usage, "]")

stream_answer("In two sentences, explain why streaming helps a chat UI.")

**The event shape (so you can parse it)** - `messageStart` -> many `contentBlockDelta` (text in `delta.text`; tool args arrive in `delta.toolUse.input`) -> `contentBlockStop` -> `messageStop` (carries `stopReason`) -> `metadata` (carries `usage`/`metrics`).

**Why** - `ConverseStream` shipped with Converse. Streaming is the difference between a UI that feels instant and one that hangs; on a hosted agent, a long non-streamed call can hit the request timeout.

### 5c - Stop reasons (read them, do not assume)

`stopReason` tells you *why* the model stopped. Treating every stop as "done" is a real bug - a truncated answer (`max_tokens`) looks complete if you ignore it.

In [ ]:
def describe_stop(reason):
    table = {
        "end_turn":            "Done. Use the answer.",
        "tool_use":            "It wants a tool. Run it and continue the loop.",
        "max_tokens":          "TRUNCATED. Raise maxTokens or continue the generation - do NOT treat as final.",
        "stop_sequence":       "Hit one of your stop sequences.",
        "guardrail_intervened":"A Guardrail blocked or modified the output. Inspect the trace.",
        "content_filtered":    "Content was filtered. Adjust the request or handle gracefully.",
    }
    return table.get(reason, "Unknown/newer stop reason - log it and inspect.")

r = get_client().converse(modelId=MODEL, messages=[{"role": "user", "content": [{"text": "Say hi."}]}],
                          inferenceConfig={"maxTokens": 64})
print("stopReason:", r["stopReason"], "->", describe_stop(r["stopReason"]))

**The full enum** (from the Converse API reference): `end_turn`, `tool_use`, `max_tokens`, `stop_sequence`, `guardrail_intervened`, `content_filtered` - plus newer additions `malformed_model_output`, `malformed_tool_use`, `model_context_window_exceeded`.

**Why** - the common break is treating `max_tokens` as a finished answer, so users get cut-off text. Branch on `stopReason` explicitly. *Evolution:* AWS keeps adding stop reasons (the `malformed_*` ones are recent), so always have an "unknown -> log it" branch.

### 5d - Throttling and retries

Agent loops are bursty - one user turn can be 5-10 model calls. New accounts often start with tiny (or zero) per-model quotas, so you will see `ThrottlingException`. Let boto3 back off for you.

In [ ]:
if not OFFLINE_DEMO:
    import boto3
    from botocore.config import Config
    retry_cfg = Config(retries={"max_attempts": 5, "mode": "adaptive"})   # adaptive = rate limiting + backoff
    rt_resilient = boto3.client("bedrock-runtime", region_name=REGION, config=retry_cfg)
    print("built a bedrock-runtime client with adaptive retries")
else:
    print("attach Config(retries={'max_attempts':5,'mode':'adaptive'}) to boto3.client('bedrock-runtime', config=...)")

**Why** - `adaptive` mode adds client-side rate limiting on top of exponential backoff, which smooths out the bursts an agent loop creates. But retries are not a substitute for quota: if your **applied quota is 0**, no amount of retrying helps - request an increase. Design for **requests-per-minute and concurrent sessions**, not daily totals, because retries + tool calls + retrieval multiply your real call rate.

## Recap

- A model call is stateless; a **conversation** is the `messages` list you carry; an **agent** is that list plus a **loop** the model drives via `tool_use`.
- Use **Converse** (since May 30, 2024); `invoke_model` is the model-specific predecessor.
- Production-readiness is four habits: **guard the loop**, **stream long output**, **branch on `stopReason`**, **let boto3 retry**.

**Next:** Demo 1 builds workflows and multi-agent on this same foundation - without a framework. Demo 2 does it with **Strands**. Demo 3 **hosts** it on **AgentCore**.